In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Load the dataset we created in Week 1
df = pd.read_csv('../data/events.csv')
print("Data loaded!")
print(df.head())

Data loaded!
    event_type       city  guest_count  budget_inr  duration_days   season  \
0  Anniversary  Hyderabad          290     1019886              3   Summer   
1    Corporate      Delhi          234     1275131              3  Monsoon   
2     Birthday  Hyderabad          277      770811              2   Winter   
3  Anniversary  Hyderabad          180      843016              2   Winter   
4      Wedding      Delhi           78     1259747              3   Winter   

   outdoor  
0     True  
1    False  
2    False  
3     True  
4    False  


In [6]:
# Machine learning only understands numbers, not text
# So we convert text columns like 'Wedding', 'Bangalore' into numbers
le = LabelEncoder()

df['event_type_encoded'] = le.fit_transform(df['event_type'])
df['city_encoded'] = le.fit_transform(df['city'])
df['season_encoded'] = le.fit_transform(df['season'])
df['outdoor_encoded'] = df['outdoor'].astype(int)

print("Before encoding:", df['event_type'].unique())
print("After encoding:", df['event_type_encoded'].unique())

Before encoding: <StringArray>
['Anniversary', 'Corporate', 'Birthday', 'Wedding', 'Party']
Length: 5, dtype: str
After encoding: [0 2 1 4 3]


In [7]:
# Define what goes IN to the model (features) and what it should predict (target)
features = ['event_type_encoded', 'city_encoded', 'guest_count', 
            'duration_days', 'season_encoded', 'outdoor_encoded']

X = df[features]  # Input
y = df['budget_inr']  # What we want to predict

# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 160
Testing samples: 40


In [8]:
# Create the model and train it
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Test it on the 20% it has never seen
y_pred = model.predict(X_test)

# Measure how good it is
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ₹{mae:,.0f}")
print(f"R2 Score: {r2:.2f}")

Mean Absolute Error: ₹82,226
R2 Score: 0.85


In [9]:
import joblib

joblib.dump(model, '../models/budget_model.pkl')
joblib.dump(features, '../models/features.pkl')

print("Model saved to models/budget_model.pkl")

Model saved to models/budget_model.pkl


In [10]:
# Simulate a real user input
# "I want to plan a Wedding in Bangalore for 300 guests, 2 days, Winter, indoors"

sample_input = pd.DataFrame([{
    'event_type_encoded': 4,   # Wedding = 4
    'city_encoded': 0,         # Bangalore = 0
    'guest_count': 300,
    'duration_days': 2,
    'season_encoded': 2,       # Winter = 2
    'outdoor_encoded': 0       # indoors = 0
}])

predicted_budget = model.predict(sample_input)[0]
print(f"Predicted budget for your event: ₹{predicted_budget:,.0f}")

Predicted budget for your event: ₹1,420,935
